# 🔥 Análise de Queimadas no Brasil
**Projeto G1 — Tema 03**  
**Aluno:** Théo Cubas Reis

## 1. Introdução
Este notebook investiga os focos de queimadas no Brasil entre 2015 e 2024 usando a base simulada fornecida para a disciplina. O objetivo é identificar tendências temporais, sazonalidade, diferenças entre regiões, estados e biomas, além de explorar a relação entre o índice de seca e os focos.

## 2. Contextualização ambiental
As queimadas podem afetar biodiversidade, qualidade do ar, saúde pública, mudanças climáticas, recursos naturais e produção agrícola. A análise procura transformar os registros da base em indicadores úteis para identificar padrões e períodos críticos.

## 3. Explicação da base
A base contém registros por ano, mês, data, região, UF, bioma, quantidade de focos, temperatura média, chuva, área atingida, índice de seca, qualidade do ar e nível de risco.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine

pd.set_option("display.max_columns", None)
df = pd.read_csv("../dados/simulacao_queimadas_brasil.csv", parse_dates=["data"])
df.head()

In [ ]:
print("Dimensões:", df.shape)
display(df.info())
display(df.describe(include="all").T)

## 4. Limpeza e preparação
Primeiro verificamos tipos, duplicidades e valores ausentes. Como a base possui os campos necessários para a análise, preservamos os registros válidos e criamos atributos derivados para facilitar a interpretação.

In [ ]:
print("Valores ausentes:")
display(df.isna().sum())
print("Duplicados:", df.duplicated().sum())

meses = ["Janeiro","Fevereiro","Março","Abril","Maio","Junho",
         "Julho","Agosto","Setembro","Outubro","Novembro","Dezembro"]
df["mes_nome"] = df["mes"].map(lambda x: meses[x-1])
df["periodo"] = df["data"].dt.to_period("M").astype(str)
df["focos_por_km2"] = df["focos_queimada"] / df["area_atingida_km2"].replace(0, np.nan)
df.head()

## 5. Engenharia de atributos
Foram criados `mes_nome`, `periodo` e `focos_por_km2`. O último indicador relaciona a quantidade de focos com a área atingida, sendo usado apenas como métrica exploratória.

## 6. Análise exploratória e KPIs

In [ ]:
total_focos = df["focos_queimada"].sum()
area_total = df["area_atingida_km2"].sum()
estado_top = df.groupby("uf")["focos_queimada"].sum().idxmax()
regiao_top = df.groupby("regiao")["focos_queimada"].sum().idxmax()
mes_top = df.groupby("mes")["focos_queimada"].sum().idxmax()
media_anual = df.groupby("ano")["focos_queimada"].sum().mean()
corr = df["indice_seca"].corr(df["focos_queimada"])

print(f"Total de focos: {total_focos:,}")
print(f"Área total atingida: {area_total:,.2f} km²")
print(f"Estado mais afetado: {estado_top}")
print(f"Região mais crítica: {regiao_top}")
print(f"Mês mais crítico: {mes_top} - {meses[mes_top-1]}")
print(f"Média anual: {media_anual:,.0f}")
print(f"Correlação seca × focos: {corr:.3f}")

### Evolução temporal

In [ ]:
anual = df.groupby("ano", as_index=False)["focos_queimada"].sum()
plt.figure(figsize=(10,5))
sns.lineplot(data=anual, x="ano", y="focos_queimada", marker="o")
plt.title("Evolução dos focos de queimadas")
plt.xlabel("Ano")
plt.ylabel("Focos")
plt.tight_layout()
plt.show()
display(anual)

**Interpretação:** a série anual permite observar os períodos de maior e menor volume de focos e serve como base para investigar mudanças no comportamento ao longo da década.

### Comparação entre regiões e estados

In [ ]:
reg = df.groupby("regiao", as_index=False)["focos_queimada"].sum().sort_values("focos_queimada", ascending=False)
uf = df.groupby("uf", as_index=False)["focos_queimada"].sum().sort_values("focos_queimada", ascending=False)

fig, axes = plt.subplots(1,2, figsize=(14,5))
sns.barplot(data=reg, x="focos_queimada", y="regiao", ax=axes[0])
axes[0].set_title("Focos por região")
sns.barplot(data=uf, x="focos_queimada", y="uf", ax=axes[1])
axes[1].set_title("Ranking de estados")
plt.tight_layout()
plt.show()

display(reg)
display(uf.head(10))

### Análise por bioma

In [ ]:
bioma = df.groupby("bioma", as_index=False)["focos_queimada"].sum().sort_values("focos_queimada", ascending=False)
plt.figure(figsize=(10,5))
sns.barplot(data=bioma, x="focos_queimada", y="bioma")
plt.title("Focos por bioma")
plt.tight_layout()
plt.show()
display(bioma)

### Sazonalidade

In [ ]:
mensal = df.groupby(["mes","mes_nome"], as_index=False)["focos_queimada"].sum().sort_values("mes")
plt.figure(figsize=(11,5))
sns.heatmap(
    df.pivot_table(index="ano", columns="mes", values="focos_queimada", aggfunc="sum"),
    annot=True, fmt=".0f", cmap="YlOrRd"
)
plt.title("Heatmap mensal de focos")
plt.xlabel("Mês")
plt.ylabel("Ano")
plt.tight_layout()
plt.show()
display(mensal)

**Interpretação:** a distribuição mensal evidencia a sazonalidade dos registros. Na base completa, outubro aparece como o mês de maior volume, seguido por julho, setembro e agosto.

### Relação entre seca e queimadas

In [ ]:
plt.figure(figsize=(9,6))
sns.regplot(data=df, x="indice_seca", y="focos_queimada", scatter_kws={"alpha":0.35})
plt.title(f"Índice de seca × focos — correlação = {corr:.3f}")
plt.xlabel("Índice de seca")
plt.ylabel("Focos")
plt.tight_layout()
plt.show()

**Interpretação:** a correlação de aproximadamente **0,815** indica uma associação positiva forte na base simulada: valores maiores do índice de seca aparecem associados a maior quantidade de focos. Esse resultado não permite afirmar causalidade.

### Nível de risco

In [ ]:
risco = df.groupby("nivel_risco", as_index=False)["focos_queimada"].sum()
display(risco)
plt.figure(figsize=(8,5))
sns.barplot(data=risco, x="nivel_risco", y="focos_queimada")
plt.title("Focos por nível de risco")
plt.tight_layout()
plt.show()

## 7. Integração com SQLite
Para atender à persistência dos dados, a base também é armazenada em SQLite utilizando SQLAlchemy.

In [ ]:
engine = create_engine("sqlite:///../database/queimadas.db")
df.drop(columns=["mes_nome","periodo","focos_por_km2"], errors="ignore").to_sql("queimadas", engine, if_exists="replace", index=False)
consulta = pd.read_sql("SELECT ano, uf, SUM(focos_queimada) AS focos FROM queimadas GROUP BY ano, uf ORDER BY focos DESC LIMIT 10", engine)
consulta

## 8. Conclusão
A análise demonstra como a base simulada pode ser usada para investigar padrões de queimadas por tempo, espaço, bioma e risco. Os principais indicadores são o total de focos, área atingida, estado e região mais afetados, mês crítico e média anual. A relação positiva entre índice de seca e focos é um dos principais achados quantitativos.

Os resultados representam **dados simulados para fins acadêmicos** e não devem ser tratados como estatísticas oficiais de queimadas no Brasil.